In [7]:
import pandas as pd
import numpy as np

In [59]:
!head ratings_long.csv

userId,movieId,rating
0,16,5
0,72,5
0,86,5
0,259,1
0,319,4
0,521,4
0,534,2
0,671,1
0,673,2


In [9]:
r = np.full((20, 1000),fill_value=np.nan)

In [10]:
df = pd.read_csv('ratings_long.csv')

In [11]:
for rec in df.itertuples():
    r[rec.userId][rec.movieId] = rec.rating

Note that $r$ matrix is $20 \times 1000$ with only <1\% full (highly sparse)

In [12]:
r

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan,  4., nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])

What if we define two matricies
- u = $20 \times 4$
- v = $4 \times 1000$


Then model $r$ as $u \times v$

Problem is we have to learn for $20 \times 4 + 4 \times 1000 = 4080$ parameters (better than 20.000 x 0.99 missing values)

## Problem

1. Define a convex loss function wrt $u$ and $v$
- Solve using gradient descent algorithm explained in **I Do**
- Use any regulatizer $L1$ or $L2$ to prevent overfitting

## Veri Özeti Çıkarma

In [1]:
%%duckdb

summarize ratings_long.csv

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,userId,BIGINT,0,19,22,8.58,6.08182067423628,3,9,14,200,0.0
1,movieId,BIGINT,5,998,206,493.17,289.9099168698744,241,489,746,200,0.0
2,rating,BIGINT,1,5,5,3.06,1.3095704923263427,2,3,4,200,0.0


In [5]:
%%duckdb

select count(rating) from ratings_long.csv group by rating 

,count(rating)
0,37
1,41
2,29
3,37
4,56


In [10]:
%%duckdb

select movieid, count(movieid),  from ratings_long.csv group by movieid order by movieid 

,movieId,count(movieid)
0,5,1
1,10,1
2,14,1
3,16,2
4,17,1
...,...,...
174,948,1
175,955,1
176,966,1
177,994,1


## Verinin eğitim ve test olarak ayrılması

In [100]:
from sklearn.model_selection import train_test_split

X_df = df[["userId", "movieId"]]
Y_df = df["rating"]


X_train, X_test, y_train, y_test = train_test_split(X_df, Y_df, test_size=0.2, stratify=df['rating'])

train_df = X_train.copy()
train_df['rating'] = y_train

test_df = X_test.copy()
test_df['rating'] = y_test

r_train = np.zeros((20,1000))

r_test = np.zeros((20, 1000))


for _, row in train_df.iterrows():
    i, j = int(row['userId']), int(row['movieId'])
    r_train[i, j] = row['rating']

for _, row in test_df.iterrows():
    i, j = int(row['userId']), int(row['movieId'])
    r_test[i, j] = row['rating']


## Loss fonksiyonu hesaplama

In [99]:
def loss_function(y: float, y_pred: float) -> float:
    return np.mean((y-y_pred) ** 2)

 ## Model eğitimi

In [ ]:
def Predicter(r_train: np.ndarray, n_epoch: int, lmbd: float, lr: float, early_stopping: bool=True):
    u = np.random.randn(20, 4)  * np.sqrt(2 / (20 + 4))
    v = np.random.randn(4, 1000) * np.sqrt(2 / (4 + 1000))
    

    for epoch in range(n_epoch):

        y_pred = u@v
        
        d_u = -2 * (r_train - u@v) @ v.T + 2 * lmbd * u
        d_v = -2 * u.T @ (r_train- u@v) + 2 * lmbd * v

        print(f"Epoch: {epoch}----Gradient norm: {np.linalg.norm(y_pred)}----- {loss_function(r_train, y_pred):.4f}")


        if np.linalg.norm(y_pred) < 0.01 and early_stopping:
            break

        u -= lr * d_u
        v -= lr * d_v

    return u, v


## Model Değerlendirmesi

In [115]:
pred = Predicter(r_test, 100, 0.01, 0.0005)


pred

Epoch: 0----Gradient norm: 3.7987057402243076----- 0.0232
Epoch: 1----Gradient norm: 3.7831743622991825----- 0.0232
Epoch: 2----Gradient norm: 3.7679474918299034----- 0.0231
Epoch: 3----Gradient norm: 3.7530206062190947----- 0.0231
Epoch: 4----Gradient norm: 3.7383893081815254----- 0.0231
Epoch: 5----Gradient norm: 3.724049322206102----- 0.0231
Epoch: 6----Gradient norm: 3.7099964911438494----- 0.0231
Epoch: 7----Gradient norm: 3.6962267729166154----- 0.0230
Epoch: 8----Gradient norm: 3.682736237341477----- 0.0230
Epoch: 9----Gradient norm: 3.669521063066088----- 0.0230
Epoch: 10----Gradient norm: 3.6565775346104----- 0.0230
Epoch: 11----Gradient norm: 3.643902039510399----- 0.0230
Epoch: 12----Gradient norm: 3.631491065559742----- 0.0229
Epoch: 13----Gradient norm: 3.6193411981452943----- 0.0229
Epoch: 14----Gradient norm: 3.607449117672817----- 0.0229
Epoch: 15----Gradient norm: 3.595811597079182----- 0.0229
Epoch: 16----Gradient norm: 3.584425499427663----- 0.0229
Epoch: 17----Gradi

(array([[ 1.81203712e-01,  2.83723290e-01,  3.29830664e-01,
         -1.76019000e-01],
        [ 1.35352754e-01, -3.12719061e-01, -4.58653691e-01,
         -1.12136295e-01],
        [-4.06620496e-01,  6.90821828e-01, -1.21652792e-01,
         -1.87890009e-01],
        [-5.63389699e-02,  1.76253691e-01, -1.90379100e-01,
          1.03178508e-01],
        [-1.48498170e-01,  5.06199630e-01, -6.31502748e-01,
         -1.74732598e-01],
        [-3.54476779e-01, -3.47511121e-01, -3.08684066e-01,
         -2.35779697e-01],
        [ 9.22810149e-02, -3.52080637e-01, -3.35509016e-01,
          1.40845150e-01],
        [ 7.88340796e-02,  1.85566057e-01, -2.78177676e-01,
         -2.44279079e-01],
        [-4.55275756e-01,  2.02112063e-01,  2.43866329e-01,
          4.11975105e-01],
        [-2.02964700e-02,  1.04089756e-01,  2.55792073e-01,
          2.62706459e-01],
        [ 3.56295628e-01,  6.96135719e-02,  3.12528182e-02,
         -2.89126320e-02],
        [-1.74620506e-01,  4.55411862e-01, 